In [ ]:
# ===================================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ===================================================================
# NFStream es la librería clave para analizar los archivos .pcap
# El resto son para manipulación de datos, IA y gráficos.
# El -q es para que la salida de la instalación sea más limpia (quiet).
# ===================================================================
!pip install -q nfstream pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# ===================================================================
# CELDA 2: IMPORTACIÓN DE MÓDULOS
# ===================================================================
import os
import pandas as pd
import numpy as np
import re
from datetime import datetime
from nfstream import NFStreamer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ignoramos advertencias para mantener la salida limpia
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")


In [ ]:
# ===================================================================
# CELDA 3: FUNCIÓN INTELIGENTE DE DESFASE
# ===================================================================
def obtener_desfase_por_csv(archivo_sync, archivo_csv):
    """
    Calcula el desfase temporal entre el timestamp del juego y el timestamp UNIX
    buscando una marca de sincronización en un archivo de log.
    """
    print(f"   -> Buscando sincronización para {archivo_csv}...")
    
    # Extrae la fecha y hora del nombre del fichero de telemetría (ej: YYYYMMDD_HHMMSS)
    match_csv = re.search(r"(\d{8}_\d{6})", archivo_csv)
    if not match_csv:
        raise ValueError("El nombre del CSV no tiene el formato de fecha esperado (YYYYMMDD_HHMMSS).")
    fecha_csv = match_csv.group(1)
    
    with open(archivo_sync, 'r') as f:
        for linea in f: # Buscar linea por linea en el archivo de sincronización
            patron = r"\[([\d\.]+)\] ==== \[SYNC_TELEMETRIA\] Tiempo interno: ([\d\.]+) ===="
            match = re.search(patron, linea) # Extrae el tiempo UNIX y el tiempo del juego
            if match:
                tiempo_unix = float(match.group(1))
                tiempo_juego_sec = float(match.group(2))
                
                # Compara si la fecha y hora del log coincide con la del archivo CSV
                fecha_unix_str = datetime.fromtimestamp(tiempo_unix).strftime("%Y%m%d_%H%M%S")
                if fecha_unix_str == fecha_csv:
                    desfase = tiempo_unix - tiempo_juego_sec
                    print(f"      [OK] Sincronización encontrada. Desfase: {desfase:.3f} s")
                    return desfase
                    
    raise ValueError(f"No se encontró en '{archivo_sync}' la fecha correspondiente al CSV '{archivo_csv}'")


In [ ]:
# ===================================================================
# CELDA 4: FUNCIÓN DE PREPROCESAMIENTO Y FUSIÓN
# ===================================================================
def preprocesar_y_fusionar(archivo_csv, pcap_local, pcap_amigo, desfase, id_host, id_amigo, tipo_red, es_ataque, fase):
    """
    Toma los datos de telemetría y red, los fusiona y enriquece para la IA.
    """
    # --- 1. PREPARAR TELEMETRÍA ---
    df_juego = pd.read_csv(archivo_csv)
    df_juego['timestamp_real'] = df_juego['timestamp'] + desfase
    df_juego = df_juego.sort_values('timestamp_real')
    print(f"   -> Telemetría leída: {len(df_juego)} filas.")

    jugadores_ids = df_juego['player_id'].unique()
    if id_host not in jugadores_ids or id_amigo not in jugadores_ids:
        print(f"      [AVISO] IDs configurados ({id_host}, {id_amigo}) no coinciden con los del CSV: {jugadores_ids}. Revisa si alguien no se conectó.")
    
    df_juego_host = df_juego[df_juego['player_id'] == id_host].copy()
    df_juego_amigo = df_juego[df_juego['player_id'] == id_amigo].copy()
    print(f"      - Datos Host (ID {id_host}): {len(df_juego_host)} filas.")
    print(f"      - Datos Amigo (ID {id_amigo}): {len(df_juego_amigo)} filas.")

    # --- Función interna para procesar PCAPs ---
    def procesar_pcap(pcap_path, tipo_conexion_str):
        print(f"   -> Procesando PCAP {tipo_conexion_str} ({pcap_path})...")
        try:
            streamer = NFStreamer(source=pcap_path, statistical_analysis=True, active_timeout=1)
            df_red = streamer.to_pandas()
            if df_red is None or df_red.empty:
                print(f"      [AVISO] El PCAP '{pcap_path}' está vacío o no contiene flujos. Se omitirá.")
                return pd.DataFrame()
            
            print(f"      - Flujos de red encontrados: {len(df_red)}.")
            df_red['tipo_conexion'] = tipo_conexion_str
            df_red['timestamp_sec'] = df_red['bidirectional_first_seen_ms'] / 1000.0
            return df_red.sort_values('timestamp_sec')
        except Exception as e:
            print(f"      [ERROR] No se pudo procesar '{pcap_path}'. Error: {e}. Se omitirá.")
            return pd.DataFrame()

    # --- 2. PROCESAR Y FUSIONAR HOST (Local) ---
    df_red_local = procesar_pcap(pcap_local, 'local')
    df_combinado_host = pd.DataFrame()
    if not df_red_local.empty and not df_juego_host.empty:
        df_combinado_host = pd.merge_asof(
            df_juego_host, df_red_local, left_on='timestamp_real', right_on='timestamp_sec', direction='nearest', tolerance=0.2
        )
        print(f"   -> Fusión Host: {len(df_combinado_host)} filas combinadas.")
    # --- 3. PROCESAR Y FUSIONAR AMIGO (Cliente) ---
    df_red_amigo = procesar_pcap(pcap_amigo, 'amigo')
    df_combinado_amigo = pd.DataFrame()
    if not df_red_amigo.empty and not df_juego_amigo.empty:
        df_combinado_amigo = pd.merge_asof(
            df_juego_amigo, df_red_amigo, left_on='timestamp_real', right_on='timestamp_sec', direction='backward', tolerance=0.2
        )#nearest daba problemas con el desfase, así que usamos backward para asegurar que tomamos el último paquete antes del evento de juego.
        print(f"   -> Fusión Amigo: {len(df_combinado_amigo)} filas combinadas.")
    else:
        # Si el PCAP del amigo falla, es un error crítico para el entrenamiento.
        raise RuntimeError(f"Fallo crítico: no se pudieron obtener datos de red para el amigo en la sesión {archivo_csv}.")

    # --- 4. JUNTAR MUNDOS ---
    df_combinado = pd.concat([df_combinado_host, df_combinado_amigo], ignore_index=True)
    df_combinado = df_combinado.ffill().fillna(0)
    
    # --- 5. INGENIERÍA DE CARACTERÍSTICAS ---
    df_combinado['delta_yaw'] = df_combinado.groupby('player_id')['yaw'].diff().abs().fillna(0)
    df_combinado['delta_pitch'] = df_combinado.groupby('player_id')['pitch'].diff().abs().fillna(0)
    df_combinado['ratio_velocidad_bytes'] = df_combinado['velocity'] / (df_combinado['bidirectional_bytes'] + 1)
    df_combinado.replace([np.inf, -np.inf], 0, inplace=True)

    # --- 6. ETIQUETADO METODOLÓGICO ---
    df_combinado['tipo_red'] = tipo_red
    df_combinado['etiqueta_real'] = -1 if es_ataque else 1
    df_combinado['fase'] = fase

    return df_combinado


In [ ]:
# ===================================================================
# CELDA 5: APRENDIZAJE AISLADO Y PREDICCIÓN GLOBAL
# ===================================================================
def entrenar_isolation_forest(df_completo):
    """
    Entrena un modelo IsolationForest usando solo datos limpios y de la fase 'train',
    y luego lo usa para predecir anomalías en todo el dataset.
    """
    print("\n[AI] Preparando datos para el entrenamiento...")
    
    # Lista de características para el modelo. Centralizarla aquí facilita su gestión.
    features = [
        # --- 1. Física del Juego (Telemetría de QuakeC) ---
        'velocity', 'delta_yaw', 'delta_pitch', 'is_attacking',
        
        # --- 2. Volumen y Dirección de Red (NFStream) ---
        'src2dst_packets', 'dst2src_packets', 
        'src2dst_bytes', 'dst2src_bytes',
        'bidirectional_duration_ms',
        
        # --- 3. Tiempos y Latencia (Detección de Lag Switches) ---
        'bidirectional_mean_piat_ms', 
        'bidirectional_stddev_piat_ms',
        
        # --- 4. Comportamiento de Paquetes (Detección de Flood) ---
        'bidirectional_mean_ps',
        'bidirectional_stddev_ps',
        
        # --- 5. Variable Sintética (Ingeniería de Características) ---
        'ratio_velocidad_bytes'
    ]
    
    # APRENDIZAJE: Solo usa el cliente, sin ataques, y de la fase "train"
    df_para_aprender = df_completo[
        (df_completo['tipo_conexion'] == 'amigo') & 
        (df_completo['etiqueta_real'] == 1) & 
        (df_completo['fase'] == 'train')
    ].copy()
    
    if len(df_para_aprender) == 0:
        raise ValueError("No hay datos de la fase 'train' (limpios y del amigo) para enseñar a la IA.")
            
    X_aprender = df_para_aprender[features].copy()
    
    # Normalizar las características para que ninguna domine sobre las otras
    scaler = StandardScaler()
    X_aprender_scaled = scaler.fit_transform(X_aprender)
    
    print("[AI] Entrenando modelo Isolation Forest con la línea base (Juego Limpio)...")
    modelo = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    modelo.fit(X_aprender_scaled)
    
    # PREDICCIÓN: Juzgamos a TODOS los jugadores y todas las fases
    print("[AI] Aplicando el modelo entrenado a todo el histórico de partidas...")
    
    # Aseguramos que el dataset completo tenga las mismas columnas que el de entrenamiento
    for f in features:
        if f not in df_completo.columns:
            df_completo[f] = 0
            
    X_total = scaler.transform(df_completo[features].copy())
    df_completo['prediccion_ia'] = modelo.predict(X_total)
    
    anomalias = len(df_completo[df_completo['prediccion_ia'] == -1])
    normales = len(df_completo[df_completo['prediccion_ia'] == 1])
    print(f"   [OK] Veredicto global: {normales} registros normales, {anomalias} anomalías marcadas.")
    
    return modelo, df_completo, features, scaler


In [ ]:
# ===================================================================
# CELDA 6: EVALUACIÓN CIENTÍFICA (TEST SET)
# ===================================================================
def evaluar_rendimiento(df_evaluacion):
    """
    Mide qué tan bien lo hizo la IA sobre los datos que nunca ha visto durante el entrenamiento.
    """
    print("\n[EVALUACIÓN] Generando métricas sobre el conjunto de TEST ciego...")
    
    # Extraemos exclusivamente los datos de la fase "test" para evaluar
    df_test = df_evaluacion[df_evaluacion['fase'] == 'test'].copy()
    
    if len(df_test) == 0:
        print("   [AVISO] No configuraste ninguna partida con fase='test'. No se puede realizar la evaluación final.")
        return
    
    y_real = df_test['etiqueta_real']
    y_predicha = df_test['prediccion_ia']
    
    print("\n=== REPORTE DE CLASIFICACIÓN (EXAMEN FINAL) ===")
    etiquetas_presentes = np.unique(np.concatenate([y_real, y_predicha]))
    nombres = []
    if -1 in etiquetas_presentes: nombres.append('Ataque (-1)')
    if 1 in etiquetas_presentes: nombres.append('Normal (1)')
    
    print(classification_report(y_real, y_predicha, labels=etiquetas_presentes, target_names=nombres, zero_division=0))
    
    matriz = confusion_matrix(y_real, y_predicha, labels=[-1, 1])
    plt.figure(figsize=(8, 6))
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predice: Ataque', 'Predice: Normal'], 
                yticklabels=['Real: Ataque', 'Real: Normal'])
    plt.title('Matriz de Confusión: Rendimiento del Anti-Cheat en el Test Set')
    plt.ylabel('Verdad Absoluta (Lo que pasó)')
    plt.xlabel('Predicción de la IA (Lo que detectó)')
    plt.tight_layout()
    
    # Guardar y mostrar la gráfica
    plt.savefig('matriz_confusion_test.png')
    print("\n   [OK] Gráfica guardada como 'matriz_confusion_test.png'")
    plt.show()


In [ ]:
# ===================================================================
# CELDA 7: CONFIGURACIÓN DEL EXPERIMENTO
# ===================================================================

# Ruta al archivo que sincroniza los tiempos.
ARCHIVO_SYNC_GLOBAL = "sincronizacion_tiempos.txt"

# SESIONES DE JUEGO
# - fase='train': Datos limpios para que la IA aprenda qué es "normal".
# - fase='test': Datos con ataques para examinar a la IA.
# ===================================================================
sesiones_a_procesar = [
    # --- FASE DE ENTRENAMIENTO (Debe ser limpia para construir la normalidad) ---
    {
        "csv": "dataset_telemetria_20260702_215841.csv",
        "pcap_local": "trafico_local.pcap",
        "pcap_amigo": "trafico_amigo.pcap",
        "id_host": 10, "id_amigo": 9,
        "tipo_red": "5G", "es_ataque": False, 
        "fase": "train"  
    },
    {
        "csv": "dataset_telemetria_20260702_220345.csv",
        "pcap_local": "trafico_local.pcap",
        "pcap_amigo": "trafico_amigo.pcap",
        "id_host": 10, "id_amigo": 9,
        "tipo_red": "5G", "es_ataque": False, 
        "fase": "train"   
    },
    {
        "csv": "dataset_telemetria_20260702_220826.csv",
        "pcap_local": "trafico_local.pcap",
        "pcap_amigo": "trafico_amigo.pcap",
        "id_host": 10, "id_amigo": 9,
        "tipo_red": "5G", "es_ataque": False, 
        "fase": "train"   
    },
    {
        "csv": "dataset_telemetria_20260702_221150.csv",
        "pcap_local": "trafico_local.pcap",
        "pcap_amigo": "trafico_amigo.pcap",
        "id_host": 10, "id_amigo": 9,
        "tipo_red": "5G", "es_ataque": False, 
        "fase": "train"   
    },
    # --- FASE DE TEST
    {
        "csv": "dataset_telemetria_20260702_221530.csv",
        "pcap_local": "trafico_local.pcap",
        "pcap_amigo": "trafico_amigo.pcap",
        "id_host": 10, "id_amigo": 9,
        "tipo_red": "5G", "es_ataque": False, 
        "fase": "test"   
    }
]


In [ ]:
# ===================================================================
# CELDA 8: EJECUCIÓN DEL PROCESO COMPLETO
# ===================================================================
lista_datasets = []
all_files_exist = True

# Verificación inicial de archivos
if not os.path.exists(ARCHIVO_SYNC_GLOBAL):
    print(f"[ERROR CRÍTICO] No se encuentra el archivo de tiempos '{ARCHIVO_SYNC_GLOBAL}'. Súbelo a Colab.")
    all_files_exist = False
else:
    for sesion in sesiones_a_procesar:
        for key in ["csv", "pcap_local", "pcap_amigo"]:
            if not os.path.exists(sesion[key]):
                print(f"[ERROR] Falta el archivo '{sesion[key]}' para la sesión. Súbelo a Colab.")
                all_files_exist = False

# Solo si todos los archivos existen, procedemos.
if all_files_exist:
    try:
        # 1. Procesamiento masivo
        for i, sesion in enumerate(sesiones_a_procesar, 1):
            print(f"\n====== PROCESANDO SESIÓN {i}/{len(sesiones_a_procesar)}: {sesion['csv']} ======")
            desfase = obtener_desfase_por_csv(ARCHIVO_SYNC_GLOBAL, sesion['csv'])
            
            df_sesion = preprocesar_y_fusionar(
                sesion['csv'], sesion['pcap_local'], sesion['pcap_amigo'], 
                desfase, sesion['id_host'], sesion['id_amigo'],
                sesion['tipo_red'], sesion['es_ataque'], sesion['fase']
            )
            lista_datasets.append(df_sesion)
            
        if not lista_datasets:
            raise ValueError("Ninguna sesión pudo ser procesada. Revisa los nombres de tus archivos y los IDs de jugador.")

        print("\n=> Uniendo todas las sesiones en un Dataset Maestro...")
        dataset_maestro = pd.concat(lista_datasets, ignore_index=True)
        
        # 2. Entrenamiento y Predicción
        print(dataset_maestro)
        modelo_entrenado, dataset_final, features, scaler = entrenar_isolation_forest(dataset_maestro)
        
        # 3. Volcado de datos para análisis posterior
        nombre_salida = "dataset_global_evaluado.csv"
        dataset_final.to_csv(nombre_salida, index=False)
        print(f"\n=> [OK] Dataset final con predicciones exportado a '{nombre_salida}'")
        
        # 4. Evaluación Científica
        evaluar_rendimiento(dataset_final)
        
        print("\n\n¡PROCESO FINALIZADO CON ÉXITO!")
        
    except Exception as e:
        print(f"\n[ERROR FATAL] La ejecución se detuvo: {e}")

